# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstitute_hdf5_from_s3.py` but uses the **local filesystem** for Parquet storage and **SQLite** for metadata. No cloud credentials needed.

**Core workflow**
1. **Ingest granules from four instruments** (GMI, SSMIS, AMSR2, ATMS) → local Parquet + SQLite metadata (four instruments so the overlap analytics in step 10 can show 2-, 3- and 4-way rendezvous)
2. **Find intersecting data** via STARE SIDs (bbox filter optional; default loads the full granule)
3. **Load intersecting Parquet partitions** from disk
4. **Reconstitute HDF5** (S1 + S2 scans)
5. **Structure comparison** — reconstituted vs original
6. **SQLite metadata verification**

**Temporal features**

7. **Temporal catalog** — every chunk carries `[t_start, t_end]` + podcode
8. **Period-filtered intersection** — data-level `[t_start, t_end]` overlap
9. **VCF temporal roll-up** — union range per pod, on the fly
10. **Multi-instrument overlap analytics** — 2-, 3- and 4-way rendezvous

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
#                       "-q"])

In [2]:
import os
import sqlite3
import h5py
from starepandas.demo_lib import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [3]:
# Parquet store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

# Resolve the sample granules from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# the pair with the STAREPODS_SAMPLE_GRANULE* env vars.
import pandas as pd
import starepandas
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
_GRANULE_DIR = os.path.join(_REPO_ROOT, "tests", "data", "granules")

GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(_GRANULE_DIR,
                 "1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5"),
)

# GMI + SSMIS are a co-located pair (both 2025-01-01, concurrent orbits) whose
# ground tracks cross within ~3 min in 42 shared pods — the tightest rendezvous
# in the demo data.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(_GRANULE_DIR,
                 "1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5"),
)

# A pair only ever fills the n=2 column of the slide-9 table. These four
# granules — one per instrument, all later the same day — are a verified
# **4-way** rendezvous: over pods q03200 and q03203 the passes arrive
# SSMIS 21:29 -> AMSR2 21:46 -> ATMS 21:47 -> GMI 22:13, i.e. all four within
# ~45 min. Ingesting them alongside the pair populates every cell of the
# slide-8 matrix and the n=2/3/4 columns of the slide-9 table from real data.
# The two windows are ~9 h apart, so they never cross-contaminate.
RENDEZVOUS_GRANULES = [
    ("SSMIS", os.path.join(_GRANULE_DIR,
              "1C.F18.SSMIS.XCAL2021-V.20250101-S195732-E213923.078446.V07B.HDF5")),
    ("AMSR2", os.path.join(_GRANULE_DIR,
              "1C.GCOMW1.AMSR2.XCAL2016-V.20250101-S201914-E215806.067167.V07A.HDF5")),
    ("ATMS",  os.path.join(_GRANULE_DIR,
              "1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5")),
    ("GMI",   os.path.join(_GRANULE_DIR,
              "1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5")),
]

# Coincidence window for step 10. The four passes above span ~45 min, so a
# narrower window still shows 2- and 3-way rendezvous but no 4-way.
OVERLAP_DT = pd.Timedelta(minutes=45)

# Steps 2-5 reconstitute GRANULE_FILE specifically, but two GMI granules are
# now ingested. This granule-level bound on "RawData Collected Time" (the
# filename-derived collection time, i.e. each granule's S... start: 11:29 vs
# 20:49) keeps those steps scoped to the first — distinct from the data-level
# `period` filter demonstrated in step 8.
PRIMARY_GRANULE_END_DATE = "2025-01-01T12:00:00"

# An instant lying in the ~7 h gap between the two GMI passes (which cover
# 11:29-13:03 and 20:49-22:22). Step 8 uses it to split the chunks by their
# data-level range.
PASS_SPLIT = pd.Timestamp("2025-01-01T16:00:00")

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/reconsitution/gmi_local_reconstituted.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstituted HDF5 (e.g. 3x the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS      : {os.path.basename(SSMIS_GRANULE_FILE)}")
for _instrument, _path in RENDEZVOUS_GRANULES:
    print(f"{_instrument:11s}: {os.path.basename(_path)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"STARE level: {STARE_LEVEL}")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

Granule    : 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5
SSMIS      : 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5
SSMIS      : 1C.F18.SSMIS.XCAL2021-V.20250101-S195732-E213923.078446.V07B.HDF5
AMSR2      : 1C.GCOMW1.AMSR2.XCAL2016-V.20250101-S201914-E215806.067167.V07A.HDF5
ATMS       : 1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5
GMI        : 1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5
Datasets   : ['GMI_S1', 'GMI_S2']
BBox       : None  (None = full granule)
STARE level: 4
Local root : /tmp/stare_pods_local
Clean first: True


## Step 1 — Ingest GMI, SSMIS, AMSR2 and ATMS granules → local Parquet + SQLite

In [4]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

Removed existing data at /tmp/stare_pods_local


In [5]:
%%time
import time
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=STARE_LEVEL)
print(f"GMI  : written {len(local_paths)} scan path(s).")

ssmis_paths = demo.ingest_granules(SSMIS_GRANULE_FILE, instrument='SSMIS', level=STARE_LEVEL)
print(f"SSMIS: written {len(ssmis_paths)} scan path(s).")

for instrument, path in RENDEZVOUS_GRANULES:
    paths = demo.ingest_granules(path, instrument=instrument, level=STARE_LEVEL)
    print(f"{instrument:5s}: written {len(paths)} scan path(s).  ({os.path.basename(path)})")

INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5 (granule=1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


INFO:starepandas.ingest:Found 1 SSMIS file(s)


...

INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5


ATMS : written 2 scan path(s).  (1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5)


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5 (granule=1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


GMI  : written 2 scan path(s).  (1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5)
CPU times: user 1min 19s, sys: 5.5 s, total: 1min 25s
Wall time: 1min 25s


## Step 2 — Find intersecting data via STARE SIDs

In [6]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all partitions will be loaded (full granule reconstitution)")

# Two GMI granules are ingested; scope steps 2-5 to the one being reconstituted.
intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'],
                                           end_date=PRIMARY_GRANULE_END_DATE)
print(f"Found {len(intersecting)} metadata row(s) for {os.path.basename(GRANULE_FILE)}.")
intersecting[['Dataset', 'grouped_id', 'group_path']]

INFO:starepandas.demo_lib:Loaded all 513 partitions for GMI


No bbox filter — all partitions will be loaded (full granule reconstitution)
Found 513 metadata row(s) for 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5.


,Dataset,grouped_id,group_path
0,GMI_S1,1819454249457680388,/tmp/stare_pods_local/q30/q302/q3022/q30220/q3...
1,GMI_S1,2206763817411543044,/tmp/stare_pods_local/q33/q331/q3311/q33110/q3...
2,GMI_S1,2269814212194729988,/tmp/stare_pods_local/q33/q333/q3330/q33300/q3...
3,GMI_S1,1826209648898736132,/tmp/stare_pods_local/q30/q302/q3022/q30223/q3...
4,GMI_S1,2276569611635785732,/tmp/stare_pods_local/q33/q333/q3330/q33303/q3...
...,...,...,...
508,GMI_S2,2254051613498933252,/tmp/stare_pods_local/q33/q332/q3322/q33221/q3...
509,GMI_S2,1778921852811345924,/tmp/stare_pods_local/q30/q301/q3011/q30112/q3...
510,GMI_S2,1776670052997660676,/tmp/stare_pods_local/q30/q301/q3011/q30111/q3...
511,GMI_S2,1781173652625031172,/tmp/stare_pods_local/q30/q301/q3011/q30113/q3...


## Step 3 — Load intersecting Parquet partitions from disk

In [7]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions found.")
    data_dict = {}

INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S1


INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S2


GMI_S1: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Tc5,Tc6,...,incidenceAngleIndex5,incidenceAngleIndex6,incidenceAngleIndex7,incidenceAngleIndex8,incidenceAngleIndex9,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.418663,-77.007088,1820290086553930635,2025-01-01 11:29:53.002,173.179993,108.080002,207.080002,159.630005,233.660004,243.289993,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012
1,-60.431023,-77.121124,1820285685235480459,2025-01-01 11:29:53.002,172.660004,106.129997,206.289993,157.559998,234.039993,241.669998,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012
2,-60.443966,-77.234947,1820286429792121643,2025-01-01 11:29:53.002,173.179993,106.599998,204.559998,155.669998,232.119995,240.059998,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012


GMI_S2: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Quality,incidenceAngle,...,sunLocalTime,incidenceAngleIndex1,incidenceAngleIndex2,incidenceAngleIndex3,incidenceAngleIndex4,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.957882,-76.767624,1820203234296677259,2025-01-01 11:29:53.002,264.320007,264.010010,250.850006,259.510010,0,49.57,...,6.319123,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012
1,-60.968994,-76.870178,1820203112318506923,2025-01-01 11:29:53.002,264.339996,263.239990,250.050003,260.750000,0,49.57,...,6.312288,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012
2,-60.980633,-76.972519,1820209139251676299,2025-01-01 11:29:53.002,267.589996,265.049988,250.809998,261.109985,0,49.57,...,6.305466,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [8]:
%%time
import time
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]

recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    granule_name=granule_basename,
)
print(f"Written to: {recon_path}")

INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over bbox=None


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over bbox=None


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/reconsitution/gmi_local_reconstituted.h5


Written to: /tmp/reconsitution/gmi_local_reconstituted.h5
CPU times: user 2.05 s, sys: 537 ms, total: 2.59 s
Wall time: 1.94 s


## Step 5 — Structure comparison: reconstituted vs original

In [9]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_local_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear    

## Step 6 — SQLite metadata verification

In [10]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} partition(s)")

SQLite DB: /tmp/stare_pods_local/metadata.db
  AMSR2_S1: 358 partition(s)
  AMSR2_S2: 357 partition(s)
  AMSR2_S3: 357 partition(s)
  AMSR2_S4: 358 partition(s)
  AMSR2_S5: 359 partition(s)
  AMSR2_S6: 356 partition(s)
  ATMS_S1: 502 partition(s)
  ATMS_S2: 501 partition(s)
  GMI_S1: 529 partition(s)
  GMI_S2: 491 partition(s)
  SSMIS_S1: 751 partition(s)
  SSMIS_S2: 750 partition(s)
  SSMIS_S3: 740 partition(s)
  SSMIS_S4: 747 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

Each ingested chunk now records its temporal range and quaternary pod code. `load_local_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) — never the heavy `MetadataJson`.

In [11]:
from starepandas.io.granules import load_local_temporal_catalog, load_local_vcf
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table, pair_drilldown,
    pod_drilldown,
)

catalog = load_local_temporal_catalog(demo.db_path)
print(f"Thin catalog: {len(catalog)} chunks across {catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog: 7156 chunks across 14 datasets


,chunks,first_start,last_end
Dataset,,,
AMSR2_S1,358,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S2,357,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S3,357,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S4,358,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S5,359,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S6,356,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
ATMS_S1,502,2025-01-01 20:17:07.136,2025-01-01 21:58:35.136
ATMS_S2,501,2025-01-01 20:17:07.136,2025-01-01 21:58:35.136
GMI_S1,529,2025-01-01 11:29:53.002,2025-01-01 22:22:22.797


,podcode,Dataset,t_start,t_end
0,q31301,SSMIS_S1,2025-01-01 11:28:14.868,2025-01-01 13:08:35.636
1,q31301,SSMIS_S3,2025-01-01 11:28:14.868,2025-01-01 13:08:35.636
2,q31301,SSMIS_S2,2025-01-01 11:28:14.868,2025-01-01 13:08:37.534
3,q31301,SSMIS_S4,2025-01-01 11:28:14.868,2025-01-01 13:08:37.534
4,q31300,SSMIS_S1,2025-01-01 11:28:14.868,2025-01-01 13:09:06.015
5,q31300,SSMIS_S2,2025-01-01 11:28:14.868,2025-01-01 13:09:06.015


## Step 8 — Period-filtered intersection

`find_intersecting_data(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period — ANDed with the spatial pod match. Two GMI passes are now ingested, ~9 h apart, so the filter can tell them apart: the same chunks, selected purely on their temporal range rather than on which granule they came from. A window days away returns none. (This is distinct from the granule-level `start_date`/`end_date` filename filter used in step 2.)

In [12]:
gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
passes = {
    "first GMI pass ": gmi[gmi['t_start'] < PASS_SPLIT],
    "second GMI pass": gmi[gmi['t_start'] >= PASS_SPLIT],
}

for label, chunks in passes.items():
    window = (chunks['t_start'].min(), chunks['t_end'].max())
    hit = demo.find_intersecting_data(None, ['GMI'], period=window)
    print(f"{label}: [{window[0]}, {window[1]}]  ({len(chunks)} chunks)")
    print(f"  -> {len(hit)} of {len(gmi)} GMI chunks match this period")

miss_period = (PASS_SPLIT - pd.Timedelta(days=10), PASS_SPLIT - pd.Timedelta(days=9))
miss = demo.find_intersecting_data(None, ['GMI'], period=miss_period)
print(f"9-10 days earlier -> {len(miss)} chunks")

INFO:starepandas.demo_lib:Loaded all 513 partitions for GMI


INFO:starepandas.demo_lib:Loaded all 507 partitions for GMI


first GMI pass : [2025-01-01 11:29:53.002000, 2025-01-01 13:03:04.224000]  (513 chunks)
  -> 513 of 1020 GMI chunks match this period
second GMI pass: [2025-01-01 20:49:11.577000, 2025-01-01 22:22:22.797000]  (507 chunks)
  -> 507 of 1020 GMI chunks match this period
9-10 days earlier -> 0 chunks


## Step 9 — VCF temporal roll-up

The temporal hierarchy ("Virtual Collection File") is queryable on the fly: `load_local_vcf(db, level)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` plus its child count. Nothing is materialized — a different level just re-groups the same thin load.

In [13]:
vcf = load_local_vcf(demo.db_path, level=1)
print(f"{len(vcf)} level-1 VCF nodes (one per octant subtree)")
vcf

32 level-1 VCF nodes (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q00,2025-01-01 21:34:43.136,2025-01-01 22:13:11.550,258,0
1,q01,2025-01-01 11:33:52.836,2025-01-01 22:19:56.548,218,0
2,q02,2025-01-01 11:50:23.955,2025-01-01 21:33:52.469,336,0
3,q03,2025-01-01 11:36:17.138,2025-01-01 22:16:30.299,508,0
4,q10,2025-01-01 20:58:55.956,2025-01-01 21:14:13.026,192,0
5,q11,2025-01-01 21:14:14.925,2025-01-01 21:15:57.455,4,0
6,q12,2025-01-01 12:26:02.360,2025-01-01 12:39:09.856,55,0
7,q13,2025-01-01 21:12:11.510,2025-01-01 21:15:09.986,12,0
8,q20,2025-01-01 12:37:07.982,2025-01-01 21:13:11.568,42,0
9,q21,2025-01-01 12:58:58.431,2025-01-01 20:58:17.199,36,0


## Step 10 — Multi-instrument overlap analytics

`rendezvous_events` sweeps the catalog for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs all aggregate that one events frame. A rendezvous means a pod where the instruments each have a chunk **and** their pass times fall within Δt — a spatial *and* temporal intersection.

With four instruments ingested, the slide-9 table gains its **n=3 and n=4** columns. The 4-way is real but tight: over pods `q03200`/`q03203` the passes arrive SSMIS 21:29 → AMSR2 21:46 → ATMS 21:47 → GMI 22:13, spanning ~45 min — so Δt has to be at least that wide before all four count as simultaneous. The first cell below shows that progression.

In [14]:
print("How the coincidence window dt widens what counts as a rendezvous:")
for dt in (pd.Timedelta(minutes=15), pd.Timedelta(minutes=30), OVERLAP_DT):
    ev = rendezvous_events(catalog, dt)
    table = overlap_pod_table(ev)
    by_n = {int(n): int(table[n].gt(0).sum()) for n in table.columns}
    print(f"  dt={str(dt).split()[-1]}  {len(ev):5d} events  "
          f"{ev['podcode'].nunique():4d} pods   pods by n-way: {by_n}")

events = rendezvous_events(catalog, OVERLAP_DT)
pod_table = overlap_pod_table(events)
widest = max(pod_table.columns)

print(f"\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8), dt={OVERLAP_DT}:")
display(overlap_matrix(events))

print('Per-pod n-way combination counts (slide 9), widest rendezvous first:')
display(pod_table.sort_values(sorted(pod_table.columns, reverse=True), ascending=False).head(10))

print(f"{int(pod_table[widest].gt(0).sum())} pods see all {widest} instruments; "
      f"{int(pod_table.get(3, pd.Series(dtype=int)).gt(0).sum())} see a 3-way.")

print('GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):')
display(pair_drilldown(events, 'GMI', 'SSMIS').head(8))

pod = pod_table[pod_table[widest].gt(0)].index[0]
print(f'Pod drill-down for {pod} — every combination meeting there:')
display(pod_drilldown(events, pod).drop(columns='times'))

How the coincidence window dt widens what counts as a rendezvous:
  dt=00:15:00   1317 events   399 pods   pods by n-way: {2: 399, 3: 8}
  dt=00:30:00   1673 events   438 pods   pods by n-way: {2: 438, 3: 96}
  dt=00:45:00   1684 events   442 pods   pods by n-way: {2: 442, 3: 101, 4: 2}

Instrument x instrument matrix — pods where A & B rendezvous (slide 8), dt=0 days 00:45:00:


,AMSR2,ATMS,GMI,SSMIS
AMSR2,0,359,43,58
ATMS,359,0,68,81
GMI,43,68,0,47
SSMIS,58,81,47,0


Per-pod n-way combination counts (slide 9), widest rendezvous first:


n_instruments,2,3,4
podcode,,,
q03200,6,4,1
q03203,6,4,1
q00101,3,1,0
q00110,3,1,0
q00111,3,1,0
q00112,3,1,0
q00113,3,1,0
q00132,3,1,0
q00220,3,1,0


2 pods see all 4 instruments; 101 see a 3-way.
GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):


,podcode,frequency,times
0,q01200,2,"[2025-01-01 22:15:32.174000, 2025-01-01 22:15:..."
1,q03200,2,"[2025-01-01 22:13:52.800000, 2025-01-01 22:14:..."
2,q03201,1,[2025-01-01 22:14:11.550000]
3,q03203,2,"[2025-01-01 22:13:39.675000, 2025-01-01 22:13:..."
4,q22200,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
5,q22201,2,"[2025-01-01 12:56:51.101000, 2025-01-01 12:56:..."
6,q22202,2,"[2025-01-01 12:55:24.851000, 2025-01-01 12:55:..."
7,q22203,4,"[2025-01-01 12:56:20.840000, 2025-01-01 12:56:..."


Pod drill-down for q03200 — every combination meeting there:


,instruments,n_instruments,frequency
0,"(AMSR2, ATMS)",2,2
1,"(AMSR2, GMI)",2,2
2,"(AMSR2, SSMIS)",2,6
3,"(ATMS, GMI)",2,2
4,"(ATMS, SSMIS)",2,2
5,"(GMI, SSMIS)",2,2
6,"(AMSR2, ATMS, GMI)",3,2
7,"(AMSR2, ATMS, SSMIS)",3,2
8,"(AMSR2, GMI, SSMIS)",3,2
9,"(ATMS, GMI, SSMIS)",3,2
